# Complete project: historical phishing triage

This optional project stands apart from the 14 modules. Use the complete course source and core environment with regression/verification extras. It requires no previous notebook state, GPU or paid service. All stages call the same implementation as projects/phishing-triage/scripts/run.py.

**Goal:** inspect how a framed question becomes a bounded tuning experiment, multi-seed evaluation, frozen final study, inference contract and completed report. No URLs are visited. Historical evidence does not authorize live blocking.

In [ ]:
from pathlib import Path
import sys, json, tempfile
import pandas as pd
candidates = [Path.cwd(), *Path.cwd().parents, Path.cwd() / "ml_ai_it"]
ROOT = next((p for p in candidates if (p / "src/practical_ml_ai").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Keep the notebook inside the complete source tree.")
sys.path.insert(0, str(ROOT / "src"))
from practical_ml_ai import capstone as study
config_path = ROOT / "projects/phishing-triage/configs/study.json"
configuration = study.config_check(study.read_json(config_path))
(ROOT / "runs").mkdir(exist_ok=True)
run_dir = Path(tempfile.mkdtemp(prefix="capstone-notebook-", dir=ROOT / "runs")) / "study"
print("New evidence directory:", run_dir)
print("Training seeds:", configuration["training_seeds"])
print("Separate split seeds:", configuration["split_seeds"])

## 1. Frame the decision and audit the data

An analyst wants a review queue, not automatic blocking. Our research questions concern model choice, training-seed sensitivity and separate development-split sensitivity. The 5:1 error-cost ratio is an illustrative assumption. Predict which decision-time feature and independence questions the source cannot answer.

Preparation preserves source attribution/hashes, removes conflicting profiles and locks a grouped outer split. Read the source feature definitions as well as the structural dictionary.

In [ ]:
study.prepare(run_dir, config_path)
audit = study.read_json(run_dir / "data_audit.json")
print(json.dumps(audit, indent=2))
print(pd.read_csv(run_dir / "training_profile.csv").head(12).to_string(index=False))

## 2. Tune within training, not on final data

Declare the candidate grid and fold boundaries before looking at scores. Predict whether a deeper tree must win. Each pipeline fits its encoder inside the fitting fold; identical profiles never cross folds. Search scores choose configurations and are not unbiased estimates of their final performance.

In [ ]:
study.tune(run_dir)
trials = pd.read_csv(run_dir / "search_trials.csv")
comparison = trials.groupby(["family", "parameters"]).balanced_accuracy.agg(["mean", "std", "count"]).reset_index()
print(comparison.to_string(index=False))
print("CV fits:", len(trials))

## 3. Measure training-seed stability

Hold data and tuned configurations fixed. Predict which model will vary most and whether an unchanged logistic result proves generalization. The procedure reports every seed and chooses a common validation threshold for each family. It never selects a best seed.

In [ ]:
study.stability(run_dir)
validation_runs = pd.read_csv(run_dir / "validation_runs.csv")
print(validation_runs[["family", "seed", "threshold", "balanced_accuracy", "cost_per_1000"]].to_string(index=False))
variation = pd.read_csv(run_dir / "validation_stability.csv")
print(variation[["family", "count", "mean", "std", "balanced_accuracy_mean", "balanced_accuracy_std"]].to_string(index=False))

## 4. Separate sensitivity to data partitioning

This optional investigation re-splits only the original training partition. Its estimator seed stays fixed. The configuration already used these observations in CV, so the study is descriptive rather than a fresh holdout. Do not combine its spread with training-seed variation as if the sources were identical.

In [ ]:
study.split_study(run_dir)
split_results = pd.read_csv(run_dir / "split_sensitivity.csv")
print(split_results[["family", "split_seed", "training_seed", "balanced_accuracy", "cost_per_1000"]].to_string(index=False))

## 5. Freeze before final evaluation

Read the chosen family, common threshold, complete seed set and ensemble rule. Predict why an ensemble is not another seed and why averaging need not improve the result. Final labels have not been parsed by the selection stages.

In [ ]:
study.freeze(run_dir)
selection = study.read_json(run_dir / "selection.json")
print({key: selection[key] for key in ["family", "threshold", "seeds", "aggregation", "selection"]})

## 6. Evaluate the declared batch and interpret uncertainty

The evaluation includes all planned seeds, their frozen probability-averaged ensemble and the baseline. No result can select a replacement. Distinguish conditional seed standard deviation from the illustrative profile-bootstrap interval. Read false positives and false negatives, not just a headline score.

In [ ]:
study.evaluate(run_dir)
evaluation = study.read_json(run_dir / "evaluation.json")
print(pd.read_csv(run_dir / "final_scores.csv")[["method", "seed", "tp", "fp", "fn", "tn", "balanced_accuracy", "cost_per_1000"]].to_string(index=False))
print(pd.read_csv(run_dir / "final_seed_stability.csv").to_string(index=False))
print(pd.read_csv(run_dir / "profile_bootstrap.csv").to_string(index=False))

## 7. Demonstrate inference, rejection and monitoring

The numeric-only bundle reproduces frozen predictions without pickle. Its exact input contract rejects missing features, target fields, booleans and unknown categories. Monitoring batches follow source order, not time. A complete project can conclude that live deployment is not justified.

In [ ]:
study.deliver(run_dir)
request = study.read_json(run_dir / "example_request.json")
bundle = study.read_json(run_dir / "inference_bundle.json")
print("Valid inference:", study.predict(bundle, request))
try:
    study.predict(bundle, {**request, "label": 1})
except ValueError as error:
    print("Expected rejection:", error)
else:
    raise AssertionError("Targets must not be accepted as inputs.")
print(pd.read_csv(run_dir / "monitoring_rehearsal.csv")[["batch", "rows", "maximum_category_rate_shift", "recall", "cost_per_1000"]].to_string(index=False))
print("Live deployment approved:", study.read_json(run_dir / "release_decision.json")["live_deployment_approved"])

## 8. Read the completed report and inspect the ledger

Now compare the annotated report with your predictions. It includes the problem, source/literature context, methods, development and final results, error interpretation, delivery decision, threats to validity and artifact guide. Successful execution alone is not evidence that a student can defend those decisions.

In [ ]:
study.report(run_dir)
study.verify(run_dir, "report")
print((run_dir / "report.md").read_text(encoding="utf-8")[:2300])
print("Complete annotated report:", run_dir / "report.md")
print("All stage markers:", sorted(p.name for p in run_dir.glob("*.done.json")))

## Independent transfer

Choose one justified development modification, record your expected effect and use a new run directory. Explain what changed in code/configuration and evidence. Repeated evaluation on the known public benchmark is reproduction/development, not a new prospective claim.

For a final capstone, provide your own research question, permitted data, source/feature audit, declared experiments, uncertainty and recommendation under the adopting course requirements. Do not copy this completed report as your submission.